SETUP

In [1]:
!pip -q install -U faiss-cpu sentence-transformers

import os, json, time, math, itertools
import numpy as np, pandas as pd, faiss, torch
from sentence_transformers import SentenceTransformer
from google.colab import drive
drive.mount("/content/drive")

BASE  = "/content/drive/MyDrive/miracl_eval"
PACKS = {"en": "/content/drive/MyDrive/miracl_en_indexes",
         "hi": "/content/drive/MyDrive/miracl_hi_indexes"}
MODEL = "intfloat/multilingual-e5-small"     # same model used to build both dense indexes

Mounted at /content/drive


Load the dense indexes and the query encoder

In [2]:
INDEX, DOCIDS = {}, {}
for lang, pack in PACKS.items():
    INDEX[lang]  = faiss.read_index(f"{pack}/dense.faiss")
    DOCIDS[lang] = json.load(open(f"{pack}/docids.json"))
    assert INDEX[lang].ntotal == len(DOCIDS[lang]), f"{lang}: index and docids.json do not match"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL, device=device)
if device == "cuda": model.half()
model.max_seq_length = 256

print(f"Query encoder : {MODEL} on {device}\n")
print(f"{'lang':<5}{'vectors':>12}{'dim':>6}{'dense.faiss (MB)':>20}{'first 3 docids':>30}")
for lang in PACKS:
    mb = os.path.getsize(f"{PACKS[lang]}/dense.faiss") / 1e6
    print(f"{lang:<5}{INDEX[lang].ntotal:>12,}{INDEX[lang].d:>6}{mb:>20,.0f}{str(DOCIDS[lang][:3]):>30}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Query encoder : intfloat/multilingual-e5-small on cuda

lang      vectors   dim    dense.faiss (MB)                first 3 docids
en      1,000,000   384               1,536      ['12#0', '12#1', '12#2']
hi        506,264   384                 778      ['10#0', '10#1', '14#0']


Load queries and qrels, and show what they contain

In [3]:
def load_set(lang, kind):
    """kind='eval': all usable dev queries | kind='hard': the 20-query suite (usable ones)"""
    E = f"{BASE}/{lang}"
    if kind == "eval":
        q = json.load(open(f"{E}/eval_queries.json", encoding="utf-8"))
        r = json.load(open(f"{E}/eval_qrels.json"))
        meta = {}
    else:
        usable = set(json.load(open(f"{E}/hard_usable.json")))
        q = {k: v for k, v in json.load(open(f"{E}/hard_queries.json", encoding="utf-8")).items() if k in usable}
        r = json.load(open(f"{E}/hard_qrels.json"))
        meta = json.load(open(f"{E}/hard_meta.json"))
    return q, r, meta

def fetch_texts(lang, ids):
    """scan passages.jsonl once and return {docid: (title, text)} for the wanted ids only"""
    ids, out = set(ids), {}
    with open(f"{PACKS[lang]}/passages.jsonl", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if d["id"] in ids:
                out[d["id"]] = (d["title"], d["text"])
                if len(out) == len(ids): break
    return out

SETS = {}
for lang in ["en", "hi"]:
    for kind in ["eval", "hard"]:
        try:
            SETS[(lang, kind)] = load_set(lang, kind)
        except FileNotFoundError:
            print(f"missing {lang}/{kind} files, skipping")

print(f"{'set':<10}{'#queries':>9}{'avg relevant/query':>20}{'avg judged-0/query':>20}")
for (lang, kind), (q, r, meta) in SETS.items():
    rel = [sum(v > 0 for v in r[x].values()) for x in q]
    non = [sum(v == 0 for v in r[x].values()) for x in q]
    print(f"{lang+'/'+kind:<10}{len(q):>9}{np.mean(rel):>20.2f}{np.mean(non):>20.2f}")

# one query, with its judged passages
for lang in ["en", "hi"]:
    q, r, _ = SETS[(lang, "eval")]
    qid = next(iter(q))
    texts = fetch_texts(lang, r[qid].keys())
    print(f"\n{'=' * 80}\n[{lang}] QUERY {qid}: {q[qid]}\n{'=' * 80}")
    for docid, lab in r[qid].items():
        t, x = texts.get(docid, ("?", "?"))
        print(f"  label={lab} ({'RELEVANT' if lab else 'judged NOT relevant'}) | {docid} | {t}\n      {x[:150]}...")

set        #queries  avg relevant/query  avg judged-0/query
en/eval         230                1.45                1.57
en/hard           9                1.67                1.67
hi/eval         350                2.15                7.83
hi/hard          20                2.80                7.00

[en] QUERY 0: Is Creole a pidgin of French?
  label=1 (RELEVANT) | 29810#23 | Texas
      The large size of Texas and its location at the intersection of multiple climate zones gives the state highly variable weather. The Panhandle of the s...
  label=0 (judged NOT relevant) | 49823#1 | Creole language
      The precise number of creole languages is not known, particularly as many are poorly attested or documented. About one hundred creole languages have a...
  label=0 (judged NOT relevant) | 49823#7 | Creole language
      Other scholars, such as Salikoko Mufwene, argue that pidgins and creoles arise independently under different circumstances, and that a pidgin need not...

[hi] QUERY 100

Search and show 5 queries per language, with the similarity score and the true label

In [4]:
def dense_search(lang, query, k=100):
    t0 = time.perf_counter()
    qv = model.encode([f"query: {query}"], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    t1 = time.perf_counter()
    scores, idx = INDEX[lang].search(qv, k)
    t2 = time.perf_counter()
    ids = [DOCIDS[lang][i] for i in idx[0]]
    return ids, scores[0], (t1 - t0) * 1000, (t2 - t1) * 1000     # encode ms, faiss ms

def show_examples(lang, n=5, k=5):
    q, r, _ = SETS[(lang, "eval")]
    picked = list(q)[:n]
    runs = {x: dense_search(lang, q[x]) for x in picked}
    need = {d for x in picked for d in runs[x][0][:k]} | {d for x in picked for d, v in r[x].items() if v > 0}
    texts = fetch_texts(lang, need)

    for x in picked:
        ids, sc, _, _ = runs[x]
        rel = {d for d, v in r[x].items() if v > 0}
        print(f"\n{'=' * 90}\n[{lang}→{lang}] QUERY {x}: {q[x]}   (relevant in qrels: {len(rel)})\n{'=' * 90}")
        for rank, (d, s) in enumerate(zip(ids[:k], sc[:k]), 1):
            lab = r[x].get(d)
            tag = "RELEVANT" if lab == 1 else ("judged 0 " if lab == 0 else "unjudged ")
            t, tx = texts[d]
            print(f"#{rank} sim={s:.3f} | qrels: {tag} | {d} | {t}\n      {tx[:140]}...")
        ranks = [i + 1 for i, d in enumerate(ids) if d in rel]
        print(f"  → relevant passage rank(s) in top-100: {ranks if ranks else 'none found'}")

show_examples("en")
show_examples("hi")


[en→en] QUERY 0: Is Creole a pidgin of French?   (relevant in qrels: 1)
#1 sim=0.891 | qrels: unjudged  | 24915#3 | Pidgin
      A pidgin differs from a creole, which is the first language of a speech community of native speakers that at one point arose from a pidgin. ...
#2 sim=0.886 | qrels: unjudged  | 49823#0 | Creole language
      A creole language, or simply creole, is a stable natural language that develops from the simplifying and mixing of different languages at a ...
#3 sim=0.883 | qrels: unjudged  | 40313#22 | Universal grammar
      The presence of creole languages is sometimes cited as further support for this theory, especially by Bickerton's controversial language bio...
#4 sim=0.882 | qrels: unjudged  | 49823#3 | Creole language
      A creole is believed to arise when a pidgin, developed by adults for use as a second language, becomes the native and primary language of th...
#5 sim=0.880 | qrels: unjudged  | 49823#27 | Creole language
      The Foreigner Talk (FT) hy

Evaluation (P@5, R@100, MAP, NDCG@5, latency, index size)

In [5]:
def query_metrics(ranked, rel):
    R = len(rel)
    p5   = sum(d in rel for d in ranked[:5]) / 5
    r100 = sum(d in rel for d in ranked[:100]) / R
    hits, s = 0, 0.0
    for i, d in enumerate(ranked[:100], 1):
        if d in rel:
            hits += 1; s += hits / i
    dcg  = sum(1 / math.log2(i + 1) for i, d in enumerate(ranked[:5], 1) if d in rel)
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(R, 5) + 1))
    return p5, r100, s / R, dcg / idcg

def dense_index_mb(lang):
    p = PACKS[lang]
    return (os.path.getsize(f"{p}/dense.faiss") + os.path.getsize(f"{p}/docids.json")) / 1e6

def evaluate(lang, q, r, label):
    dense_search(lang, next(iter(q.values())))                 # warm-up (not timed)
    per_q, enc, srch = {}, [], []
    for qid, text in q.items():
        ids, _, te, ts = dense_search(lang, text)
        enc.append(te); srch.append(ts)
        per_q[qid] = query_metrics(ids, {d for d, v in r[qid].items() if v > 0})
    m = np.mean(list(per_q.values()), axis=0)
    return per_q, {"Model": label, "n_queries": len(q), "P@5": m[0], "R@100": m[1], "MAP": m[2],
                   "NDCG@5": m[3], "Latency (ms)": np.mean(enc) + np.mean(srch),
                   "Encode (ms)": np.mean(enc), "FAISS (ms)": np.mean(srch), "Index (MB)": dense_index_mb(lang)}

rows, PER_Q = [], {}
for (lang, kind), (q, r, meta) in SETS.items():
    per_q, row = evaluate(lang, q, r, f"E5 dense {lang}→{lang} [{kind}]")
    rows.append(row); PER_Q[(lang, kind)] = per_q
    if kind == "hard":
        for cat in sorted(set(meta.values())):
            ids = [x for x in per_q if meta[x] == cat]
            m = np.mean([per_q[x] for x in ids], axis=0)
            rows.append({"Model": f"   └ {cat}", "n_queries": len(ids),
                         "P@5": m[0], "R@100": m[1], "MAP": m[2], "NDCG@5": m[3]})

df = pd.DataFrame(rows)
pd.set_option("display.width", 220)
display(df.round(3))

os.makedirs(f"{BASE}/results", exist_ok=True)
df.to_csv(f"{BASE}/results/dense_results.csv", index=False)

,Model,n_queries,P@5,R@100,MAP,NDCG@5,Latency (ms),Encode (ms),FAISS (ms),Index (MB)
0,E5 dense en→en [eval],230,0.203,0.967,0.607,0.636,272.049,14.690,257.359,1547.585
1,E5 dense en→en [hard],9,0.222,1.000,0.457,0.511,259.475,13.518,245.957,1547.585
2,└ homonym_ambiguity,3,0.133,1.000,0.179,0.222,NaN,NaN,NaN,NaN
3,└ negation,3,0.200,1.000,0.639,0.640,NaN,NaN,NaN,NaN
4,└ temporal,3,0.333,1.000,0.552,0.672,NaN,NaN,NaN,NaN
5,E5 dense hi→hi [eval],350,0.230,0.904,0.477,0.503,139.320,13.906,125.414,783.770
6,E5 dense hi→hi [hard],20,0.270,0.892,0.451,0.432,138.376,13.891,124.485,783.770
7,└ homonym_ambiguity,7,0.257,0.952,0.524,0.471,NaN,NaN,NaN,NaN
8,└ negation,7,0.171,0.881,0.359,0.343,NaN,NaN,NaN,NaN
9,└ temporal,6,0.400,0.833,0.473,0.492,NaN,NaN,NaN,NaN
